In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")
api_key = os.getenv("DART_API_KEY")
print(f"API 키 로드: {'✅' if api_key else '❌'}")

API 키 로드: ✅


In [ ]:
# 삼성전자 2024년 사업보고서 (연결재무제표) 호출
# corp_code: 00126380 (Day 2에서 매핑한 값)
# bsns_year: 2024 (사업연도)
#  reprt_code: 11011 (사업보고서 = 4분기, 연간)
# 💡 reprt_code 코드표:
# 11011: 사업보고서 (4분기, 연간)
# 11014: 3분기보고서
# 11012: 반기보고서 (2분기)
# 11013: 1분기보고서
# fs_div: CFS (연결재무제표)

url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
params = {
    "crtfc_key": api_key,
    "corp_code": "00126380",
    "bsns_year": "2026",
    "reprt_code": "11013",  # 사업보고서
    "fs_div": "CFS",        # 연결재무제표
}

response = requests.get(url, params=params)
data = response.json()

print(f"응답 상태: {data.get('status')}")
print(f"메시지: {data.get('message')}")
print(f"항목 수: {len(data.get('list', []))}")

응답 상태: 000
메시지: 정상
항목 수: 215


In [ ]:
# list 부분을 DataFrame으로 변환
# 💡 핵심 컬럼 의미:
# sj_nm: 재무제표 종류 (손익계산서/재무상태표/현금흐름표)
# account_id: K-IFRS 표준 계정 코드 (영문)
# account_nm: 계정 이름 (한글)
# thstrm_amount: 당기금액 (이번 기간)
# frmtrm_amount: 전기금액 (전년 동기)
# df = pd.DataFrame(data["list"])

print(f"전체 행 수: {len(df)}")
print(f"\n=== 컬럼 ===")
print(df.columns.tolist())
print(f"\n=== 재무제표 종류 분포 ===")
print(df["sj_nm"].value_counts())

전체 행 수: 215

=== 컬럼 ===
['rcept_no', 'reprt_code', 'bsns_year', 'corp_code', 'sj_div', 'sj_nm', 'account_id', 'account_nm', 'account_detail', 'thstrm_nm', 'thstrm_amount', 'frmtrm_nm', 'frmtrm_amount', 'ord', 'currency', 'thstrm_add_amount', 'frmtrm_q_nm', 'frmtrm_q_amount', 'frmtrm_add_amount']

=== 재무제표 종류 분포 ===
sj_nm
자본변동표      98
재무상태표      49
현금흐름표      38
손익계산서      17
포괄손익계산서    13
Name: count, dtype: int64


In [7]:
# 7개 원천 데이터 검색
targets = {
    "매출액": ["ifrs-full_Revenue", "ifrs_Revenue"],
    "영업이익": ["dart_OperatingIncomeLoss"],
    "당기순이익": ["ifrs-full_ProfitLoss"],
    "자본총계": ["ifrs-full_Equity"],
    "부채총계": ["ifrs-full_Liabilities"],
    "현금성자산": ["ifrs-full_CashAndCashEquivalents"],
    "감가상각비": ["dart_DepreciationExpense", "dart_DepreciationAndAmortisationExpense"],
}

print("=" * 70)
for name, account_ids in targets.items():
    found = df[df["account_id"].isin(account_ids)]
    if not found.empty:
        for _, row in found.iterrows():
            amount = row["thstrm_amount"]
            sj = row["sj_nm"]
            print(f"✅ {name:8s} | {sj:15s} | {row['account_id']:50s} | {amount}")
    else:
        print(f"❌ {name:8s} | 계정 ID 없음 → 다른 ID 탐색 필요")
print("=" * 70)

✅ 매출액      | 손익계산서           | ifrs-full_Revenue                                  | 133873444000000
✅ 영업이익     | 손익계산서           | dart_OperatingIncomeLoss                           | 57232797000000
✅ 당기순이익    | 손익계산서           | ifrs-full_ProfitLoss                               | 47225272000000
✅ 당기순이익    | 포괄손익계산서         | ifrs-full_ProfitLoss                               | 47225272000000
✅ 당기순이익    | 현금흐름표           | ifrs-full_ProfitLoss                               | 47225272000000
✅ 당기순이익    | 자본변동표           | ifrs-full_ProfitLoss                               | 0
✅ 당기순이익    | 자본변동표           | ifrs-full_ProfitLoss                               | 47101190000000
✅ 당기순이익    | 자본변동표           | ifrs-full_ProfitLoss                               | 0
✅ 당기순이익    | 자본변동표           | ifrs-full_ProfitLoss                               | 47101190000000
✅ 당기순이익    | 자본변동표           | ifrs-full_ProfitLoss                               | 124082000000
✅ 당기순이익    | 자본변동표           | ifrs-f

In [8]:
# 현금흐름표만 추출
cf = df[df["sj_nm"] == "현금흐름표"]
print(f"현금흐름표 항목 수: {len(cf)}")

# 감가상각 관련 키워드로 검색
print("\n=== '감가' 포함 항목 ===")
depreciation = cf[cf["account_nm"].str.contains("감가|상각|Depreciation|Amortis", na=False, case=False)]
print(depreciation[["account_id", "account_nm", "thstrm_amount"]].to_string())

현금흐름표 항목 수: 38

=== '감가' 포함 항목 ===
Empty DataFrame
Columns: [account_id, account_nm, thstrm_amount]
Index: []


In [9]:
def get_financial_data(corp_code: str, year: int, reprt_code: str, fs_div: str = "CFS") -> dict:
    """
    DART에서 단일 회사의 재무제표를 가져와 7개 원천 데이터를 추출한다.
    
    Args:
        corp_code: DART 기업 고유번호 (8자리)
        year: 사업연도 (예: 2024)
        reprt_code: 보고서 코드
            - 11011: 사업보고서 (4분기, 연간)
            - 11014: 3분기보고서
            - 11012: 반기보고서 (2분기)
            - 11013: 1분기보고서
        fs_div: CFS(연결) 또는 OFS(별도)
    
    Returns:
        dict: {매출액, 영업이익, 당기순이익, 감가상각비, 자본총계, 부채총계, 현금성자산}
              각 값은 정수(원 단위) 또는 None (데이터 없음)
    """
    url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
    params = {
        "crtfc_key": api_key,
        "corp_code": corp_code,
        "bsns_year": str(year),
        "reprt_code": reprt_code,
        "fs_div": fs_div,
    }
    
    response = requests.get(url, params=params, timeout=10)
    data = response.json()
    
    # 응답 검증
    if data.get("status") != "000":
        return {
            "error": f"DART 응답 오류: {data.get('status')} - {data.get('message')}",
            "corp_code": corp_code,
            "year": year,
            "reprt_code": reprt_code,
        }
    
    df = pd.DataFrame(data["list"])
    
    # 계정 ID 매핑 (회사마다 다를 수 있어 후보 여러 개)
    target_ids = {
        "매출액": ["ifrs-full_Revenue", "ifrs_Revenue"],
        "영업이익": ["dart_OperatingIncomeLoss"],
        "당기순이익": ["ifrs-full_ProfitLoss"],
        "자본총계": ["ifrs-full_Equity"],
        "부채총계": ["ifrs-full_Liabilities"],
        "현금성자산": ["ifrs-full_CashAndCashEquivalents"],
        "감가상각비": [
            "dart_DepreciationAndAmortisationExpense",
            "dart_DepreciationExpense",
        ],
    }
    
    result = {
        "corp_code": corp_code,
        "year": year,
        "reprt_code": reprt_code,
        "fs_div": fs_div,
    }
    
    for name, ids in target_ids.items():
        found = df[df["account_id"].isin(ids)]
        if not found.empty:
            # thstrm_amount는 문자열 (콤마 포함) → 정수 변환
            amount_str = found.iloc[0]["thstrm_amount"]
            try:
                # 콤마 제거 + 빈 문자열 처리
                amount = int(amount_str.replace(",", "")) if amount_str else None
            except (ValueError, AttributeError):
                amount = None
            result[name] = amount
        else:
            result[name] = None
    
    return result


# 함수 테스트 — 삼성전자 2024년 사업보고서
samsung_2024 = get_financial_data(
    corp_code="00126380",
    year=2024,
    reprt_code="11011",
    fs_div="CFS",
)

print("=== 삼성전자 2024년 (연결) ===")
for k, v in samsung_2024.items():
    if isinstance(v, int):
        print(f"  {k:12s}: {v:>20,} 원")
    else:
        print(f"  {k:12s}: {v}")

=== 삼성전자 2024년 (연결) ===
  corp_code   : 00126380
  year        :                2,024 원
  reprt_code  : 11011
  fs_div      : CFS
  매출액         :  300,870,903,000,000 원
  영업이익        :   32,725,961,000,000 원
  당기순이익       :   34,451,351,000,000 원
  자본총계        :  402,192,070,000,000 원
  부채총계        :  112,339,878,000,000 원
  현금성자산       :   53,705,579,000,000 원
  감가상각비       : None


In [10]:
# 최근 4분기 데이터 수집
# 2024년: 1Q, 2Q, 3Q, 사업보고서(연간) = 4개 보고서
# 분기별 데이터를 모두 가져온다

quarters = [
    ("2024", "11013", "2024_1Q"),  # 2024 1분기
    ("2024", "11012", "2024_2Q"),  # 2024 반기 (1~2분기 누적)
    ("2024", "11014", "2024_3Q"),  # 2024 3분기 (1~3분기 누적)
    ("2024", "11011", "2024_FY"),  # 2024 사업보고서 (연간)
]

print("4개 보고서 수집 중...\n")
results = []
for year, reprt_code, label in quarters:
    print(f"[{label}] 수집 중...", end=" ")
    data = get_financial_data("00126380", int(year), reprt_code)
    if "error" in data:
        print(f"❌ {data['error']}")
    else:
        print("✅")
    data["label"] = label
    results.append(data)

# DataFrame으로 정리
df_results = pd.DataFrame(results)
display_cols = ["label", "매출액", "영업이익", "당기순이익", "자본총계", "감가상각비"]
print(f"\n=== 분기별 데이터 ===")
print(df_results[display_cols].to_string())

4개 보고서 수집 중...

[2024_1Q] 수집 중... ✅
[2024_2Q] 수집 중... ✅
[2024_3Q] 수집 중... ✅
[2024_FY] 수집 중... ✅

=== 분기별 데이터 ===
     label              매출액            영업이익           당기순이익             자본총계 감가상각비
0  2024_1Q   71915601000000   6606009000000   6754708000000  371916124000000  None
1  2024_2Q   74068302000000  10443878000000   9841345000000  383526671000000  None
2  2024_3Q   79098731000000   9183371000000  10100904000000  386281363000000  None
3  2024_FY  300870903000000  32725961000000  34451351000000  402192070000000  None


In [11]:
# 검증: 우리가 가져온 값이 누적인지 단독인지 확인
# 단서: 손익계산서 항목의 frmtrm_nm, frmtrm_amount (전기 비교)

# 2024 3분기 보고서를 다시 직접 호출해서 원본 응답 확인
url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
params = {
    "crtfc_key": api_key,
    "corp_code": "00126380",
    "bsns_year": "2024",
    "reprt_code": "11014",  # 3분기보고서
    "fs_div": "CFS",
}

response = requests.get(url, params=params)
data = response.json()
df_raw = pd.DataFrame(data["list"])

# 매출액 행만 보기
revenue_rows = df_raw[df_raw["account_id"] == "ifrs-full_Revenue"]
print("=== 3분기보고서의 매출액 관련 행 ===")
print(revenue_rows[["sj_nm", "account_nm", "thstrm_nm", "thstrm_amount", "frmtrm_nm", "frmtrm_amount"]].to_string())

=== 3분기보고서의 매출액 관련 행 ===
    sj_nm account_nm   thstrm_nm   thstrm_amount frmtrm_nm frmtrm_amount
67  손익계산서        매출액  제 56 기 3분기  79098731000000       NaN           NaN


In [20]:
def get_financial_data_v5(
    corp_code: str,
    year: int,
    reprt_code: str,
    fs_div: str = "CFS",
) -> dict:
    """
    v5 업데이트:
    - 배당금 추가 (현금흐름표에서 발견)
    - 영업활동현금흐름 → EBITDA 대체
    """
    url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
    params = {
        "crtfc_key": api_key,
        "corp_code": corp_code,
        "bsns_year": str(year),
        "reprt_code": reprt_code,
        "fs_div": fs_div,
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as e:
        return {"error": f"네트워크 오류: {e}"}
    
    if data.get("status") != "000":
        return {"error": f"DART {data.get('status')}: {data.get('message')}"}
    
    df = pd.DataFrame(data["list"])
    
    target_ids = {
        "매출액": ["ifrs-full_Revenue", "ifrs_Revenue"],
        "영업이익": ["dart_OperatingIncomeLoss"],
        "당기순이익": ["ifrs-full_ProfitLoss"],
        "자본총계": ["ifrs-full_Equity"],
        "부채총계": ["ifrs-full_Liabilities"],
        "현금성자산": ["ifrs-full_CashAndCashEquivalents"],
        "감가상각비": [
            "dart_DepreciationAndAmortisationExpense",
            "dart_DepreciationExpense",
        ],
        # 신규 추가
        "영업활동현금흐름": [
            "ifrs-full_CashFlowsFromUsedInOperatingActivities",
            "dart_CashFlowsFromOperatingActivities",
        ],
        "배당금": [
            "ifrs-full_DividendsPaidClassifiedAsFinancingActivities",
            "ifrs-full_DividendsPaid",
            "dart_DividendsPaid",
        ],
        "자기주식취득": [
            "ifrs-full_PurchaseOfTreasuryShares",
        ],
    }
    
    result = {
        "corp_code": corp_code,
        "year": year,
        "reprt_code": reprt_code,
        "fs_div": fs_div,
    }
    
    for name, ids in target_ids.items():
        found = df[df["account_id"].isin(ids)]
        if not found.empty:
            amount_str = found.iloc[0]["thstrm_amount"]
            try:
                amount = int(amount_str.replace(",", "")) if amount_str and amount_str.strip() else None
                # 배당금/자기주식취득은 음수일 수 있음 → 절대값 사용
                if amount and name in ["배당금", "자기주식취득"]:
                    amount = abs(amount)
                result[name] = amount
            except (ValueError, AttributeError):
                result[name] = None
        else:
            result[name] = None
    
    # EBITDA 계산
    if result["감가상각비"] is not None and result["영업이익"] is not None:
        result["EBITDA"] = result["영업이익"] + result["감가상각비"]
        result["EBITDA_방식"] = "정공법"
    elif result["영업활동현금흐름"] is not None:
        result["EBITDA"] = result["영업활동현금흐름"]
        result["EBITDA_방식"] = "근사치 (영업활동현금흐름)"
    else:
        result["EBITDA"] = None
        result["EBITDA_방식"] = "계산 불가"
    
    return result


# 삼성전자 재테스트
print("=" * 60)
print("삼성전자 재무 데이터 v5 (배당금 + EBITDA 대체)")
print("=" * 60)
fin = get_financial_data_v5("00126380", 2024, "11011")
for k, v in fin.items():
    if isinstance(v, int):
        print(f"  {k:18s}: {v:>20,}")
    else:
        print(f"  {k:18s}: {v}")

삼성전자 재무 데이터 v5 (배당금 + EBITDA 대체)
  corp_code         : 00126380
  year              :                2,024
  reprt_code        : 11011
  fs_div            : CFS
  매출액               :  300,870,903,000,000
  영업이익              :   32,725,961,000,000
  당기순이익             :   34,451,351,000,000
  자본총계              :  402,192,070,000,000
  부채총계              :  112,339,878,000,000
  현금성자산             :   53,705,579,000,000
  감가상각비             : None
  영업활동현금흐름          :   72,982,621,000,000
  배당금               :   10,888,749,000,000
  자기주식취득            :    1,811,775,000,000
  EBITDA            :   72,982,621,000,000
  EBITDA_방식         : 근사치 (영업활동현금흐름)


In [14]:
def get_shares_outstanding(corp_code: str, year: int, reprt_code: str) -> dict:
    """
    DART '주식의 총수 현황' API로 발행주식수와 자기주식수를 가져온다.
    
    Returns:
        dict: {보통주_발행, 보통주_자기주식, 우선주_발행, 보통주_유통}
        - 보통주_유통 = 보통주_발행 - 보통주_자기주식 (EPS 계산용)
    """
    url = "https://opendart.fss.or.kr/api/stockTotqySttus.json"
    params = {
        "crtfc_key": api_key,
        "corp_code": corp_code,
        "bsns_year": str(year),
        "reprt_code": reprt_code,
    }
    
    response = requests.get(url, params=params, timeout=10)
    data = response.json()
    
    if data.get("status") != "000":
        return {"error": f"DART {data.get('status')}: {data.get('message')}"}
    
    df = pd.DataFrame(data["list"])
    
    result = {
        "corp_code": corp_code,
        "year": year,
        "보통주_발행": None,
        "보통주_자기주식": None,
        "보통주_유통": None,
        "우선주_발행": None,
    }
    
    # se 필드(증권 종류)로 보통주/우선주 구분
    # isu_stock_totqy: 발행주식 총수
    # distb_stock_co: 유통주식 수
    # tesstk_co: 자기주식 수
    
    for _, row in df.iterrows():
        se = row.get("se", "")
        if "보통주" in se:
            try:
                issued = int(row["isu_stock_totqy"].replace(",", "")) if row.get("isu_stock_totqy") else 0
                treasury = int(row["tesstk_co"].replace(",", "")) if row.get("tesstk_co") else 0
                result["보통주_발행"] = issued
                result["보통주_자기주식"] = treasury
                result["보통주_유통"] = issued - treasury
            except (ValueError, AttributeError):
                pass
        elif "우선주" in se:
            try:
                issued = int(row["isu_stock_totqy"].replace(",", "")) if row.get("isu_stock_totqy") else 0
                result["우선주_발행"] = issued
            except (ValueError, AttributeError):
                pass
    
    return result


# 삼성전자 테스트
print("=== 삼성전자 발행주식수 (2024년 사업보고서) ===")
shares = get_shares_outstanding("00126380", 2024, "11011")
for k, v in shares.items():
    if isinstance(v, int):
        print(f"  {k}: {v:>20,} 주")
    else:
        print(f"  {k}: {v}")

=== 삼성전자 발행주식수 (2024년 사업보고서) ===
  corp_code: 00126380
  year:                2,024 주
  보통주_발행:       20,000,000,000 주
  보통주_자기주식:           29,700,000 주
  보통주_유통:       19,970,300,000 주
  우선주_발행:        5,000,000,000 주


In [15]:
def get_current_price(stock_code: str) -> dict:
    """
    종목코드(KRX 6자리)로 최근 종가와 시가총액을 가져온다.
    
    여러 데이터 소스를 순서대로 시도 (fallback 전략).
    """
    # 시도 1: FinanceDataReader
    try:
        import FinanceDataReader as fdr
        from datetime import datetime, timedelta
        
        # 최근 7일 데이터 시도 (휴장일 고려)
        end_date = datetime.now()
        start_date = end_date - timedelta(days=7)
        df = fdr.DataReader(stock_code, start=start_date, end=end_date)
        
        if not df.empty:
            latest = df.iloc[-1]
            return {
                "stock_code": stock_code,
                "date": df.index[-1].strftime("%Y-%m-%d"),
                "close": int(latest["Close"]),
                "volume": int(latest.get("Volume", 0)),
                "source": "FinanceDataReader",
            }
    except Exception as e:
        print(f"  FDR 실패: {e}")
    
    # 시도 2: yfinance (대체 수단)
    try:
        import yfinance as yf
        ticker = yf.Ticker(f"{stock_code}.KS")  # 코스피
        hist = ticker.history(period="5d")
        if hist.empty:
            ticker = yf.Ticker(f"{stock_code}.KQ")  # 코스닥
            hist = ticker.history(period="5d")
        
        if not hist.empty:
            latest = hist.iloc[-1]
            return {
                "stock_code": stock_code,
                "date": hist.index[-1].strftime("%Y-%m-%d"),
                "close": int(latest["Close"]),
                "volume": int(latest["Volume"]),
                "source": "yfinance",
            }
    except Exception as e:
        print(f"  yfinance 실패: {e}")
    
    return {"error": "모든 주가 소스 실패", "stock_code": stock_code}


# 삼성전자 주가 테스트
print("=== 삼성전자 최근 주가 ===")
price = get_current_price("005930")
for k, v in price.items():
    if isinstance(v, int):
        print(f"  {k}: {v:>15,} 원")
    else:
        print(f"  {k}: {v}")

=== 삼성전자 최근 주가 ===
  stock_code: 005930
  date: 2026-05-26
  close:         299,000 원
  volume:      21,773,154 원
  source: FinanceDataReader


In [21]:
def calculate_metrics_v4(
    financials: dict,
    shares_data: dict,
    price: int,
) -> dict:
    """
    v4: 배당금을 financials에서 자동으로 가져옴
    """
    REQUIRED = ["당기순이익", "매출액", "자본총계"]
    missing = [k for k in REQUIRED if financials.get(k) is None]
    if missing:
        return {"error": f"필수 데이터 누락: {missing}"}
    
    shares = shares_data.get("보통주_유통")
    if not shares or not price:
        return {"error": "주식수 또는 주가 없음"}
    
    # 배당금 자동 가져옴
    total_dividend = financials.get("배당금")
    
    # 주당 지표
    eps = financials["당기순이익"] / shares
    sps = financials["매출액"] / shares
    bps = financials["자본총계"] / shares
    dps = (total_dividend / shares) if total_dividend else None
    
    # 배당성향
    dividend_payout_ratio = (
        (total_dividend / financials["당기순이익"] * 100)
        if total_dividend else None
    )
    
    # 비율 지표
    per = price / eps if eps > 0 else None
    psr = price / sps if sps > 0 else None
    pbr = price / bps if bps > 0 else None
    roe = financials["당기순이익"] / financials["자본총계"] * 100
    
    # EV/EBITDA
    ev_ebitda = None
    market_cap = price * shares
    
    can_calc_ev = (
        financials.get("EBITDA") is not None and
        financials.get("부채총계") is not None and
        financials.get("현금성자산") is not None
    )
    
    if can_calc_ev:
        net_debt = financials["부채총계"] - financials["현금성자산"]
        ev = market_cap + net_debt
        ebitda = financials["EBITDA"]
        ev_ebitda = ev / ebitda if ebitda > 0 else None
    
    return {
        "EPS": round(eps, 2),
        "SPS": round(sps, 2),
        "BPS": round(bps, 2),
        "DPS": round(dps, 2) if dps else None,
        "배당성향(%)": round(dividend_payout_ratio, 2) if dividend_payout_ratio else None,
        "PER": round(per, 2) if per else None,
        "PSR": round(psr, 2) if psr else None,
        "PBR": round(pbr, 2) if pbr else None,
        "ROE(%)": round(roe, 2),
        "EV/EBITDA": round(ev_ebitda, 2) if ev_ebitda else None,
        "_EBITDA_방식": financials.get("EBITDA_방식"),
        "_시가총액": int(market_cap),
        "_주가": price,
        "_총배당금": total_dividend,
    }


# 통합 테스트
print("\n" + "=" * 60)
print("📊 삼성전자 10개 지표 (v4)")
print("=" * 60)

fin = get_financial_data_v5("00126380", 2024, "11011")
shares = get_shares_outstanding("00126380", 2024, "11011")
price_data = get_current_price("005930")

metrics = calculate_metrics_v4(fin, shares, price_data["close"])

for k, v in metrics.items():
    if k.startswith("_"):
        continue
    print(f"  {k:15s}: {v}")

print(f"\n--- 메타 ---")
print(f"  EBITDA 방식: {metrics.get('_EBITDA_방식')}")
print(f"  주가: {metrics.get('_주가', 0):,} 원")
print(f"  시가총액: {metrics.get('_시가총액', 0) / 1e12:.1f} 조원")
print(f"  총배당금: {(metrics.get('_총배당금') or 0) / 1e12:.2f} 조원")


📊 삼성전자 10개 지표 (v4)
  EPS            : 1725.13
  SPS            : 15065.92
  BPS            : 20139.51
  DPS            : 545.25
  배당성향(%)        : 31.61
  PER            : 173.32
  PSR            : 19.85
  PBR            : 14.85
  ROE(%)         : 8.57
  EV/EBITDA      : 82.62

--- 메타 ---
  EBITDA 방식: 근사치 (영업활동현금흐름)
  주가: 299,000 원
  시가총액: 5971.1 조원
  총배당금: 10.89 조원


In [23]:
# 1. 재무 데이터 (2024년 사업보고서)
print("[1/3] 재무 데이터 수집...")
fin = get_financial_data_v5("00126380", 2024, "11011")
print(f"  당기순이익: {fin['당기순이익']:>20,} 원")
print(f"  매출액:    {fin['매출액']:>20,} 원")
print(f"  자본총계:   {fin['자본총계']:>20,} 원")

# 2. 발행주식수
print("\n[2/3] 발행주식수 수집...")
shares = get_shares_outstanding("00126380", 2024, "11011")
print(f"  보통주_유통: {shares['보통주_유통']:>15,} 주")

# 3. 주가
print("\n[3/3] 최근 주가 수집...")
price_data = get_current_price("005930")
print(f"  주가: {price_data['close']:,} 원 ({price_data['date']})")

# 4. 10개 지표 계산
print("\n" + "=" * 60)
print("📊 삼성전자 10개 지표 (2024 사업보고서 기준)")
print("=" * 60)

metrics = calculate_metrics(
    financials=fin,
    shares_data=shares,
    price=price_data["close"],
    total_dividend=None,  # 배당은 다음 단계에서
)

for k, v in metrics.items():
    if k.startswith("_"):
        continue  # 참고값은 건너뜀
    print(f"  {k:15s}: {v}")

[1/3] 재무 데이터 수집...
  당기순이익:   34,451,351,000,000 원
  매출액:     300,870,903,000,000 원
  자본총계:    402,192,070,000,000 원

[2/3] 발행주식수 수집...
  보통주_유통:  19,970,300,000 주

[3/3] 최근 주가 수집...
  주가: 299,000 원 (2026-05-26)

📊 삼성전자 10개 지표 (2024 사업보고서 기준)
  error          : 필수 데이터 누락: ['감가상각비']


전체 항목 수: 213

=== sj_nm (재무제표 종류) 분포 ===
sj_nm
자본변동표      91
재무상태표      52
현금흐름표      40
손익계산서      17
포괄손익계산서    13
Name: count, dtype: int64

=== '감가/상각/Depre' 포함 모든 항목 ===
     sj_nm                                       account_id            account_nm thstrm_amount
4    재무상태표  ifrs-full_CurrentFinancialAssetsAtAmortisedCost           단기상각후원가금융자산             0
100  현금흐름표                                     -표준계정코드 미사용-  단기상각후원가금융자산의 순감소(증가)  620858000000


현금흐름표 항목 수: 40

=== 현금흐름표 상위 항목 ===
                                                                            account_id                account_nm    thstrm_amount
82                                    dart_CashAndCashEquivalentsAtBeginningOfPeriodCf                기초현금및현금성자산   69080893000000
83                                          dart_CashAndCashEquivalentsAtEndOfPeriodCf                기말현금및현금성자산   53705579000000
84                                                                        -표준계정코드 미사용-                    매각예정분류                0
85                                    ifrs-full_CashFlowsFromUsedInFinancingActivities                  재무활동현금흐름   -7797243000000
86                                                 dart_ProceedsFromLongTermBorrowings                 장기차입금의 차입     404954000000
87                                                                        -표준계정코드 미사용-                 비지배지분의 증감       8511000000
88                                                    